In [ ]:
# Import packages
!pip install xmmpysas --upgrade
!pip install pandas stingray

import os
import re
import glob
import pysas
import shutil
import stingray
import subprocess
import numpy as np
import scipy as sc
from astropy.io import fits
from astropy.time import Time
from astropy.table import QTable
from astropy.stats import sigma_clip
from astroquery.heasarc import Heasarc
from SciServer import Authentication as auth
from heasoftpy import HSPTask, ftools, heatools, heasptools
from pysas import ObsID, MyTask
pysas.print_version.print_sas_version()

In [ ]:
# EPIC-pn Processing Function
def XMM_Newton(object_name, ObsIDs):
	
	# Create folders for the object's data
	persistent_folder = os.path.join('/home/idies/workspace/Storage/', auth.getKeystoneUserWithToken(auth.getToken()).userName,
									 f'persistent/XMM-Newton/{object_name}/')
	scratch_folder    = os.path.join('/home/idies/workspace/Temporary/', auth.getKeystoneUserWithToken(auth.getToken()).userName,
									 f'scratch/XMM-Newton/{object_name}/')
	image_dir         = os.path.join(persistent_folder, 'Images')
	spectra_dir       = os.path.join(persistent_folder, 'Spectra')
	light_curves_dir  = os.path.join(persistent_folder, 'Light Curves')
	os.makedirs(image_dir, exist_ok=True)
	os.makedirs(spectra_dir, exist_ok=True)
	os.makedirs(light_curves_dir, exist_ok=True)
	
	# ObsID selection
	if len(ObsIDs) == 0:
		Table = Heasarc.query_region(position=SkyCoord.from_name(object_name), mission='xmmmaster', radius=15*u.arcminute,
									 fields='name,obsid,time,ra,dec,pn_time,pn_mode,pps_flag,process_status,status,data_in_heasarc')
		Filtered = Table[(Table['STATUS']=="archived ") & (Table['DATA_IN_HEASARC']=="Y") & (Table['PPS_FLAG']=="Y") & (Table['PN_TIME']>=1000)]
		Filtered.sort(keys='time')
		ObsIDs = Filtered['obsid'].tolist()
	
	# ObsIDs iteration loop
	for i in ObsIDs:
		obsid, dt = str(i[0]), float(i[1])
		# Step 1: Data accessing, data calibration, & filtering
		ObsID(obsid=obsid).basic_setup(data_dir=scratch_folder, repo='sciserver', overwrite=True, rerun=True, recalibrate=True,
									   run_rgsproc=False, run_epchain=True, run_emchain=False, run_emproc=False, odfingest_opts=['oalcheck=yes'])
		Workdir = os.path.join(scratch_folder,obsid,'work')
		os.chdir(Workdir) # Change to workdir for SAS tasks
		try: MyTask('epchain', ['odfaccess=all', 'datamode=ALL', 'keepintermediate=all', 'usecanonicalnames=no', 'optloadingimage=Y', 'withradmon=yes',
								'runradmonfix=yes', 'guessdeltap=yes', 'witheventmap=Y', 'mipdist=Y', 'windowfilter=Y', 'runepreject=yes', 'withoffsetlist=yes',
								'withxrlcorrection=Y', 'withsoftflarescreening=Y', 'softflaresmooth=FLARE', 'runepnoise=yes', 'savemasks=Y', 'runepxrlcorr=yes',
								'withxrlimage=yes', 'withphotonmap=Y', 'withoutoftime=Y', 'withctisrcpos=Y', 'withccdoffsets=Y', 'withgainff=Y', 'withphagaincolumn=yes',
								'checksasmip=Y', 'screenrejected=Y', 'propagatecolumns=all', 'runepfast=Y', 'withpatplot=yes', 'runbackground=Y', 'withmask=N',
								f'rateset=OOT-bkglc_{obsid}.fits', f'specset=OOT-bkgspec_{obsid}.fits', f'outset=OOT-EL_{obsid}.fits']).run()
		except Exception as e: print(f"An unexpected error occurred during epchain for OOT-EL_{obsid}.fits: {e}")
		try: MyTask('epchain', ['odfaccess=all', 'datamode=ALL', 'keepintermediate=all', 'usecanonicalnames=no', 'optloadingimage=Y', 'withradmon=yes',
								'runradmonfix=yes', 'guessdeltap=yes', 'witheventmap=Y', 'mipdist=Y', 'windowfilter=Y', 'runepreject=yes', 'withoffsetlist=yes',
								'withxrlcorrection=Y', 'withsoftflarescreening=Y', 'softflaresmooth=FLARE', 'runepnoise=yes', 'savemasks=Y', 'runepxrlcorr=yes',
								'withxrlimage=yes', 'withphotonmap=Y', 'withoutoftime=N', 'withctisrcpos=Y', 'withccdoffsets=Y', 'withgainff=Y', 'withphagaincolumn=yes',
								'checksasmip=Y', 'screenrejected=Y', 'propagatecolumns=all', 'runepfast=Y', 'withpatplot=yes', 'runbackground=Y', 'withmask=N',
								f'rateset=Normal-bkglc_{obsid}.fits', f'specset=Normal-bkgspec_{obsid}.fits', f'outset=Normal-EL_{obsid}.fits']).run()
		except Exception as e: print(f"An unexpected error occurred during epchain for Normal-EL_{obsid}.fits: {e}")
		SAS = glob.glob('*.SAS')[0]
		GTI = glob.glob('gti*.dat')[0]
		pn_mode = fits.getheader(filename=f'Normal-EL_{obsid}.fits', extname='EVENTS')['SUBMODE']
		if   pn_mode == "PrimeSmallWindow"   : Filter_wheel, Δt, OOT_fraction = 'pn_closed_SW_2022_v2.fits.gz',  350, 1.1/100
		elif pn_mode == "PrimeLargeWindow"   : Filter_wheel, Δt, OOT_fraction = 'pn_closed_LW_2022_v2.fits.gz',  200, 0.16/100
		elif pn_mode == "PrimeFullWindow"    : Filter_wheel, Δt, OOT_fraction = 'pn_closed_FF_2022_v2.fits.gz',  100, 6.3/100
		elif pn_mode == "PrimeExtendedWindow": Filter_wheel, Δt, OOT_fraction = 'pn_closed_EFF_2022_v2.fits.gz', 100, 2.3/100
		
		# Step 2: Barycentric correction
		MyTask('barycen', [f"table=Normal-EL_{obsid}.fits:EVENTS", 'processgtis=true', 'processexposutables=true', 'ephemeris=DE405']).run()
		MyTask('barycen', [f"table=OOT-EL_{obsid}.fits:EVENTS",    'processgtis=true', 'processexposutables=true', 'ephemeris=DE405']).run()
		
		# Proton flaring filtering (tentative)
		if pn_mode != "PrimeSmallWindow":
			if pn_mode == "PrimeLargeWindow": print("EPN PrimeLargeWindow SUBMODE may produce suspect results", '\n', "Corrective action: attempts analysis")
			MyTask('espfilt', [f"eventfile=OOT-EL_{obsid}.fits", 'withoot=yes', 'method=histogram', 'withsmoothing=yes', 'withbinning=yes', 'keepinterfiles=no']).run()
			os.remove(os.path.join(Workdir,f'OOT-EL_{obsid}.fits'))
			EL = glob.glob('*-allevc.fits')[0]
			os.rename(EL, f'OOT-EL_{obsid}.fits')
			MyTask('espfilt', [f"eventfile=Normal-EL_{obsid}.fits", 'withoot=yes', 'method=histogram', 'withsmoothing=yes', 'withbinning=yes', 'keepinterfiles=no']).run()
			os.remove(os.path.join(Workdir,f'Normal-EL_{obsid}.fits'))
			EL = glob.glob('*-allevc.fits')[0]
			os.rename(EL, f'Normal-EL_{obsid}.fits')
		
		# Step 3: Source detection
		MyTask('edetect_stack', [f'eventsets=Normal-EL_{obsid}.fits', 'attitudesets=atthk.dat', f'summarysets={SAS}', f'srclistset=SourceList_{obsid}.fits',
								 'minstage=1', 'maxstage=12', 'informational=all', 'compress=true', 'pimin=300', 'pimax=10000', 'finalize=true', 'runattcalc=true',
								 'runevselectimages=true', 'ev_imagedatatype=Real64', 'runeexpmap=true', 'runemask=true', 'runeboxdetectlocal=true',
								 'runesplinemap=true', 'esp_fitmethod=spline', 'esp_withcheeseimage=true', 'esp_withcheesemask=true', 'runeboxdetectmap=true',
								 'runesensmap=true', 'runemosaic=true', 'runeboxdetectstack=true', 'runemldetect=true', 'eml_psfmodel=ellbeta', 'eml_fitextent=true',
								 'eml_extentmodel=beta', 'eml_fitposition=true', 'eml_determineerrors=true', 'eml_withsourcemap=true', 'eml_withthreshold=true',
								 'eml_threshcolumn=LIKE', 'eml_withhotpixelfilter=true', 'eml_withtwostage=true', 'eml_withxidband=true', 'eml_xidfixed=true']).run()
		MyTask('srcmatch', [f'outputlistset=SourceList_{obsid}.fits', f'htmloutput=SourceList_{obsid}.html', f'inputlistsets=SourceList_{obsid}.fits.gz',
							'usecrossidsets=no', 'omlistset=-', 'allepicmode=yes', 'extentunit=sky']).run()  # eposcorr/catorr require OM source list to be performed.
		#MyTask('esources', [f'eventsets=Normal-EL_{obsid}.fits', f'srctab=SourceList_{obsid}.fits:SRCLIST', 'setflags=yes']).run()    #... depends on srcmatch
		MyTask('poscorr3xmm', [f'srclist=SourceList_{obsid}.fits:SRCLIST']).run()
		
		# Step 4: Image extraction (SKIPPED for now due to internal problem in `eimageget`)
		try:
			MyTask('eimageget', [f'evtfile=Normal-EL_{obsid}.fits', f'ootfile=OOT-EL_{obsid}.fits', f'gtifile={GTI}', #'withfwcimages=true',
								 #f'fwcfile=/home/idies/workspace/Storage/Sulthon_Araska/persistent/XMM-Newton/Filter_wheel/{Filter_wheel}',
								 'attfile=atthk.dat', 'pimin=300', 'pimax=10000', 'patmin=0', 'patmax=4', 'withemtaglenoise=true', 'withbadpixupdate=true',
								 'withwindowmode=true', 'withattcalc=no', 'withexposure=true', 'withmask=true']).run()
			MyTask('eimagecombine', [f"prefix=PN_{obsid}", "withcheckinput=true", "maskindividual=true", "withasmooth=true", "smoothstyle=adaptive", "keepinterstage=false"]).run()
			MyTask('implot', [f"set=PN_{obsid}_ima_0_subdiv_smooth.fits", "withsrclisttab=yes", f"srclisttab=SourceList_{obsid}.fits.gz:SRCLIST",
							  "radiusstyle=psf", "withellipse=yes", "withlabels=yes", "labelstyle=rownumber", "imagestyle=image", "zscaletype=linear",
							  "colourmap=2", "srccolour=0", "trimborder=yes", "withframe=yes", "withxyclip=yes", "gridstyle=ticks", "device=/GIF"]).run()
			os.rename("pgplot.gif", f"Image-PN_{obsid}.gif")
			shutil.copyfile(os.path.join(Workdir,f'PN_{obsid}_ima_0_subdiv_smooth.fits'), os.path.join(image_dir,f'Image-PN_{obsid}.fits'))
		except Exception as e:
			print("An error occurred:", '\n', e)
			MyTask('implot', ["set=mosaic_EPIC.fits.gz", "withsrclisttab=yes", f"srclisttab=SourceList_{obsid}.fits.gz:SRCLIST",
							  "radiusstyle=psf", "withellipse=yes", "withlabels=yes", "labelstyle=rownumber", "imagestyle=image", "zscaletype=linear",
							  "colourmap=2", "srccolour=0", "trimborder=yes", "withframe=yes", "withxyclip=yes", "gridstyle=ticks", "device=/GIF"]).run()
			os.rename("pgplot.gif", f"Image-PN_{obsid}.gif")
			shutil.copyfile(os.path.join(Workdir,'mosaic_EPIC.fits.gz'), os.path.join(image_dir,f'Image-PN_{obsid}.fits'))
		shutil.copyfile(os.path.join(Workdir,f'Image-PN_{obsid}.gif'), os.path.join(image_dir,f'Image-PN_{obsid}.gif'))
		
		# Step 5: Region analysis (https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/uhb/onaxisxraypsf.html)
		Source = QTable.read(f'SourceList_{obsid}.fits', format='fits', hdu='SRCLIST')[0]  # Identified source with highest flux in the frame
		RA,DEC = Source['RA'].value, Source['DEC'].value
		image, background, exposure = glob.glob('iimage*PN_*.fits.gz')[0], glob.glob('bkgmap*PN_*.fits.gz')[0], glob.glob('expunv*PN_*.fits.gz')[0]
		eregionanalyse = subprocess.run(['eregionanalyse', f'imageset={image}', f'bkgimageset={background}', f'exposuremap={exposure}','psfmodel=ELLBETA',
										 f'srcexp=(RA,DEC) IN CIRCLE({RA},{DEC},{40/3600})'], capture_output=True, text=True, check=True).stdout
		#Src_radius = list(map(float, re.findall(r"-?\d+\.\d+|-?\d+", next(l for l in eregionanalyse.splitlines() if "optellip:" in l)) ))
		#Source_Region = f'(RA,DEC) in ELLIPS({RA},{DEC},{Src_radius[0]/3600},{Src_radius[2]/3600},{Src_radius[4]})'  #Somehow invalid in etimeget
		Src_radius = float(re.search(r"optradius: ([\d\.]+) arcsecs ([\d\.]+) image units", eregionanalyse).group(1))
		Source_Region = f'(RA,DEC) in CIRCLE({RA},{DEC},{Src_radius/2/3600})'
		ebkgreg = subprocess.run(['ebkgreg', 'imageset=mosaic_EPIC.fits.gz', 'withsrclist=false', f'srclisttab=SourceList_{obsid}.fits',
								  'withcoords=true', 'coordtype=EQPOS', f'x={RA}', f'y={DEC}', f'r={Src_radius/2}'],
								 capture_output=True, text=True, check=True).stdout
		Bkg_center = re.search(r"RA, Dec \(deg\.\)\s+: ([\d\.]+), ([\d\.]+)", ebkgreg)
		Bkg_radius = float(re.search(r"Extraction radius \(arcsec\)\s+: ([\d\.]+)", ebkgreg).group(1))
		Bckgnd_Region = f"(RA,DEC) in CIRCLE({Bkg_center.group(1)},{Bkg_center.group(2)},{Bkg_radius/3600})"
		open(f'Region_{obsid}.reg', mode='w').write("ICRS" + '\n' + Source_Region.removeprefix('(RA,DEC) in ') + "\n" 
													+ Bckgnd_Region.removeprefix('(RA,DEC) in ') + " # background")
		shutil.copyfile(os.path.join(Workdir,f'Region_{obsid}.reg'), os.path.join(image_dir,f'Region_{obsid}.reg'))
		
		# Light curve extraction & correction function
		def LightCurve(E_min, E_max, with_double=True):
			if with_double == True: max_pattern = 4
			else: max_pattern = 0  #Single-pattern only
			# Source and background count rate generation
			MyTask('etimeget', [f"table=Normal-EL_{obsid}.fits:EVENTS", "withfilestem=yes", f"filestem=IT-lc_{obsid}_{E_min}-{E_max}",
								f"srcexp=#XMMEA_EP && (PATTERN<={max_pattern}) && ({Source_Region}) && (PI in [{E_min}:{E_max}])",
								f"backexp=#XMMEA_EP && (PATTERN<={max_pattern}) && ({Bckgnd_Region}) && (PI in [{E_min}:{E_max}])",
								"timebinstyle=parameter", f"timebinwidth={dt}"]).run()
			MyTask('etimeget', [f"table=OOT-EL_{obsid}.fits:EVENTS", "withfilestem=yes", f"filestem=OT-lc_{obsid}_{E_min}-{E_max}",
								f"srcexp=#XMMEA_EP && (PATTERN<={max_pattern}) && ({Source_Region}) && (PI in [{E_min}:{E_max}])",
								f"backexp=#XMMEA_EP && (PATTERN<={max_pattern}) && ({Bckgnd_Region}) && (PI in [{E_min}:{E_max}])",
								"timebinstyle=parameter", f"timebinwidth={dt}"]).run()
			# Out-of-time correction
			ftools.lcmath(infile=f'IT-lc_{obsid}_{E_min}-{E_max}_srcrate.lc', bgfile=f'OT-lc_{obsid}_{E_min}-{E_max}_srcrate.lc', noprompt=True,
						  outfile=f'Src-lc_{obsid}_{E_min}-{E_max}.fits', multi=1, multb=OOT_fraction, docor=True, err_mode=1, type='d')
			ftools.lcmath(infile=f'IT-lc_{obsid}_{E_min}-{E_max}_bgdrate.lc', bgfile=f'IT-lc_{obsid}_{E_min}-{E_max}_bgdrate.lc', noprompt=True,
						  outfile=f'Bkg-lc_{obsid}_{E_min}-{E_max}.fits', multi=1, multb=OOT_fraction, docor=True, err_mode=1, type='d')
			# Source-background correction
			Src_hdr = fits.getheader(filename=f'Src-lc_{obsid}_{E_min}-{E_max}.fits', extname='RATE').copy()
			Src_lc  = QTable.read(f'Src-lc_{obsid}_{E_min}-{E_max}.fits', format='fits', hdu='RATE')
			for col in Src_lc.colnames: Src_lc[col] = Src_lc[col].astype('d')
			fits.update(filename=f'Src-lc_{obsid}_{E_min}-{E_max}.fits', extname='RATE', data=Src_lc, header=Src_hdr)
			Bkg_hdr = fits.getheader(filename=f'Bkg-lc_{obsid}_{E_min}-{E_max}.fits', extname='RATE').copy()
			Bkg_lc  = QTable.read(f'Bkg-lc_{obsid}_{E_min}-{E_max}.fits', format='fits', hdu='RATE')
			for col in Bkg_lc.colnames: Bkg_lc[col] = Bkg_lc[col].astype('d')
			fits.update(filename=f'Bkg-lc_{obsid}_{E_min}-{E_max}.fits', extname='RATE', data=Bkg_lc, header=Bkg_hdr)
			MyTask('epiclccorr', [f'srctslist=Src-lc_{obsid}_{E_min}-{E_max}.fits', f'bkgtslist=Bkg-lc_{obsid}_{E_min}-{E_max}.fits',
								  f'eventlist=Normal-EL_{obsid}.fits', f'outset=LightCurve_{obsid}_{E_min}-{E_max}.fits',
								  'allcamera=yes', 'withbkgset=yes', 'applyabsolutecorrections=true']).run()
			MyTask('elcplot', [f'set=LightCurve_{obsid}_{E_min}-{E_max}.fits', f'plotfile=LightCurve_{obsid}_{E_min}-{E_max}.gif',
							   f"tscaleoffset={fits.getheader(f'LightCurve_{obsid}_{E_min}-{E_max}.fits', ext=1)['MJDREF']}",
							   "binsize=1", "plotdevice=/GIF", "bkgdyscale=yes"]).run()
			os.remove(f"IT-lc_{obsid}_{E_min}-{E_max}_srcrate.lc")
			os.remove(f"OT-lc_{obsid}_{E_min}-{E_max}_srcrate.lc")
			os.remove(f"IT-lc_{obsid}_{E_min}-{E_max}_bgdrate.lc")
			os.remove(f"OT-lc_{obsid}_{E_min}-{E_max}_bgdrate.lc")
		
		# Step 6: Light curves extraction...
		Energy_bin = [300, 500, 1000, 2000, 4500, 7000, 10000] #eV
		for i in range(len(Energy_bin)):
			if i+1<len(Energy_bin): E_min, E_max = Energy_bin[i], Energy_bin[i+1]
			else: E_min, E_max = Energy_bin[0], Energy_bin[-1]  # Full energy band
			LightCurve(E_min=E_min, E_max=E_max)
		# Light curves merging, plot, and Fourier transformation.
		MyTask('elcbuild', [f"sets=LightCurve_{obsid}_300-10000.fits LightCurve_{obsid}_300-500.fits LightCurve_{obsid}_500-1000.fits LightCurve_{obsid}_1000-2000.fits \
								   LightCurve_{obsid}_2000-4500.fits LightCurve_{obsid}_4500-7000.fits LightCurve_{obsid}_7000-10000.fits",
							f"outset=LightCurve_{obsid}.fits"]).run()
		ftools.faddcol(infile=f"LightCurve_{obsid}.fits[RATE]", colfile=f"LightCurve_{obsid}_300-10000.fits[RATE]", colname='TIME', history=True)
		heatools.ftappend(infile=f"LightCurve_{obsid}_300-10000.fits[SRC_GTIS]", outfile=f"LightCurve_{obsid}.fits", history=True)
		heatools.ftappend(infile=f"LightCurve_{obsid}_300-10000.fits[BKG_GTIS]", outfile=f"LightCurve_{obsid}.fits", history=True)
		heatools.ftappend(infile=f"LightCurve_{obsid}_300-10000.fits[REG00103]", outfile=f"LightCurve_{obsid}.fits", history=True)
		MyTask('lcplot', [f"set=LightCurve_{obsid}.fits", f"plotfile=LightCurve_{obsid}.gif", "plotdevice=/GIF", "binsize=1", "bkgdyscale=yes", "tests=both"]).run()
		for i in glob.glob(f"LightCurve_{obsid}.gif*"): os.remove(i)
		try:
			MyTask('efftplot', [f"infile=LightCurve_{obsid}.fits", "bkgsub=yes", "normalization=2", "fillgaps=yes", "npoints=4", "removetrend=yes",
								"polyorder=4", "operation=1", "rebin=yes", "xscale=log", "yscale=log", f"plotdevice=/GIF", f"outfile=PDS_{obsid}.gif"]).run()
			if os.path.exists(f"PDS_{obsid}.gif_2")==True: os.rename(f"PDS_{obsid}.gif_2", f"PDS_{obsid}.gif")
			shutil.copyfile(os.path.join(Workdir,f'PDS_{obsid}.pdf'), os.path.join(light_curves_dir,f'PDS_{obsid}.pdf'))
		except Exception as e: print(f"An error occurred during efftplot task: {e}")
		shutil.copyfile(os.path.join(Workdir,f'LightCurve_{obsid}.fits'), os.path.join(light_curves_dir,f'LightCurve_{obsid}.fits'))
		
		# Step 7: Corrected, binned, reduced source event list
		eventlists = []
		Energy_bands = np.geomspace(300,10000,16)
		for E_min,E_max in zip(Energy_bands, Energy_bands[1:]):
			LightCurve(E_min=E_min, E_max=E_max)
			filename = f"LightCurve_{obsid}_{E_min}-{E_max}.fits"
			Header = fits.getheader(filename, ext=1)
			lc = QTable.read(filename, format='fits', hdu=1)
			lc = lc[lc['FRACEXP']!=0]  # Delete rows with masked values
			evl = stingray.EventList.from_lc(stingray.Lightcurve(time=lc['TIME'].value, counts=lc['RATE'].value, err=lc['ERROR'].value, bg_counts=lc['BACKV'].value,
																 frac_exp=lc['FRACEXP'].value, input_counts=False, err_dist='poisson',
																 gti=QTable.read(filename, format='fits', hdu='SRC_GTIS').to_pandas().to_numpy(),
																 dt=Header['TIMEDEL'], header=Header, mission='XMM-Newton', instr='EPIC-pn'))
			evl.energy   = [sc.stats.gmean([Header['CHANMIN']/1E3, Header['CHANMAX']/1E3])]*len(evl)
			evl.header   = Header
			evl.dt       = Header['TIMEDEL']
			evl.mjdref   = Header['MJDREF']
			evl.timeref  = 'SOLARSYSTEM'
			evl.timesys  = 'TDB'
			evl.rmf_file = f'Spectra/RMF_{obsid}.rmf'
			evl.mission  = "XMM-Newton"
			evl.instr    = "EPIC-pn"
			evl.detector_id = "XMM"
			evl.high_precision = True
			eventlists.append(evl)
			os.remove(filename)
			if os.path.exists(f"LightCurve_{obsid}_{E_min}-{E_max}.gif")==True: os.remove(f"LightCurve_{obsid}_{E_min}-{E_max}.gif")
		EVL = eventlists[0].join(other=eventlists[1:], strategy='union')
		EVL.energy = np.array(EVL.energy)
		EVL.write(filename=f'EVL-PN_{obsid}.fits', fmt='fits')
		HDUL = fits.open(f'Normal-EL_{obsid}.fits').copy()
		HDUL['PRIMARY'].header.set(keyword='CREATOR', value='Stingray-2.2.10')
		HDUL['PRIMARY'].header.set(keyword='DATE', value=Time.now().to_value('isot'))
		HDUL['PRIMARY'].header.remove(keyword='HISTORY', ignore_missing=True, remove_all=True)
		HDUL['PRIMARY'].header.remove(keyword='XPROC0', ignore_missing=True, remove_all=False)
		HDUL['EVENTS'].data = fits.getdata(f'EVL-PN_{obsid}.fits', ext=1)
		HDUL['EVENTS'].update_header()
		fits.HDUList(hdus=[HDUL['PRIMARY'], HDUL['EVENTS'], HDUL['STDGTI04']]).writeto(fileobj=os.path.join(persistent_folder, f'EVL-PN_{obsid}.fits'),
																					   output_verify='fix+warn', overwrite=True, checksum=True)
		
		# Step 8: Background flare filtering (https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-filterbackground-in-python)
		LightCurve(E_min=10000, E_max=12000, with_double=False)
		#flare = sigma_clip(data=QTable.read(f"LightCurve_{obsid}_1000-12000.fits", hdu=1)['BACKV'].value, cenfunc='mean', sigma=3, return_bounds=True)[-1]
		MyTask('tabgtigen', [f"table=LightCurve_{obsid}_1000-12000.fits[RATE]", "gtiset=gti04.dat:STDGTI04",
							 f"expression=(BACKV<=0.05)"]).run()
		fits.setval(filename="gti04.dat", extname='STDGTI04', keyword='TIMESYS', value="TDB")
		fits.setval(filename="gti04.dat", extname='STDGTI04', keyword='TIMEREF', value="SOLARSYSTEM")
		MyTask('evselect', [f"table=Normal-EL_{obsid}.fits", "keepfilteroutput=yes", "expression=GTI(gti04.dat,TIME)", "writedss=true",
							"updateexposure=true", "filterexposure=true"]).run()
		MyTask('evselect', [f"table=OOT-EL_{obsid}.fits", "keepfilteroutput=yes", "expression=GTI(gti04.dat,TIME)", "writedss=true",
							"updateexposure=true", "filterexposure=true"]).run()
		
		# Step 9: Spectrum extraction
		# Generate out-of-time spectral set & background correction
		MyTask('especget', [f'table=OOT-EL_{obsid}.fits:EVENTS', 'withfilestem=false', f'srcspecset=Src-Spec_{obsid}_OT.fits',
							f'bckspecset=bkg-Spec_{obsid}_OT.fits', f'srcarfset=ARF_{obsid}_OT.arf', f'srcrmfset=RMF_{obsid}_OT.rmf',
							'useodfatt=yes', 'extendedsource=no', f"srcexp=#XMMEA_EP && (PATTERN<=4) && ({Source_Region})",
							f"backexp=#XMMEA_EP && (PATTERN<=4) && ({Bckgnd_Region})"]).run()
		MyTask('backcorr', [f'bkgspectrumset=bkg-Spec_{obsid}_OT.fits', f'srcspectrumset=Src-Spec_{obsid}_OT.fits',
							f'outbkgspectrumset=Bkg-Spec_{obsid}_OT.fits', 'witheventlist=yes', f'eventlist=OOT-EL_{obsid}.fits']).run()
		os.remove(os.path.join(Workdir,f'bkg-Spec_{obsid}_OT.fits'))
		# Generate in-time spectral set & background correction
		MyTask('especget', [f'table=Normal-EL_{obsid}.fits:EVENTS', 'withfilestem=false', f'srcspecset=Src-Spec_{obsid}.fits',
							f'bckspecset=bkg-Spec_{obsid}.fits', f'srcarfset=ARF_{obsid}.arf', f'srcrmfset=RMF_{obsid}.rmf',
							'useodfatt=yes', 'extendedsource=no', f"srcexp=#XMMEA_EP && (PATTERN<=4) && ({Source_Region})",
							f"backexp=#XMMEA_EP && (PATTERN<=4) && ({Bckgnd_Region})"]).run()
		MyTask('backcorr', [f'bkgspectrumset=bkg-Spec_{obsid}.fits', f'srcspectrumset=Src-Spec_{obsid}.fits',
							f'outbkgspectrumset=Bkg-Spec_{obsid}.fits', 'witheventlist=yes', f'eventlist=Normal-EL_{obsid}.fits']).run()
		os.remove(os.path.join(Workdir,f'bkg-Spec_{obsid}.fits'))
		# [Intermezzo] Spectrum and overlayed image plots
		MyTask('efluxer', [f"spectrumset=Src-Spec_{obsid}.fits", f"backgndset=Bkg-Spec_{obsid}.fits",
						   f"arfset=ARF_{obsid}.arf", f"rmfset=RMF_{obsid}.rmf", f"fluxedset=SED_{obsid}.fits"]).run()
		try: MyTask('especplot', [f"srcspectrumset=Src-Spec_{obsid}.fits", f"bkgspectrumset=Bkg-Spec_{obsid}.fits",
								  f"plotfile=Spectrum_{obsid}.gif", "plotdevice=/GIF",]).run()
		except Exception as e: print(f"An error occurred during executing task especplot: {e}")
		fits.setval(filename=f"Src-Spec_{obsid}.fits", ext=3, keyword='EXTNAME', value="REGION")
		fits.setval(filename=f"Bkg-Spec_{obsid}.fits", ext=3, keyword='EXTNAME', value="REGION")
		try: MyTask('implotregions', ["imageset=mosaic_EPIC.fits.gz", f"outputfile=Image-PN_{obsid}_overlayed.png",  #Segmentation fault, core dumped
									  f"srcregfile=Src-Spec_{obsid}.fits", f"bkgregfile=Bkg-Spec_{obsid}.fits"]).run()
		except Exception as e: print(f"An error occured during executing implotregions task: {e}")
		# Source spectrum's out-of-time events correction:
		OOT_source  = QTable.read(f'Src-Spec_{obsid}_OT.fits', format='fits', hdu='SPECTRUM')
		Norm_source = QTable.read(f'Src-Spec_{obsid}.fits',    format='fits', hdu='SPECTRUM')
		Norm_source.replace_column(name='COUNTS', col=(Norm_source['COUNTS']-OOT_source['COUNTS']*OOT_fraction))
		spec_hdr    = fits.getheader(filename=f'Src-Spec_{obsid}.fits', extname='SPECTRUM').copy()
		fits.update(filename=f'Src-Spec_{obsid}.fits', extname='SPECTRUM', data=Norm_source.as_array(), header=spec_hdr)
		# Background spectrum's out-of-time events correction:
		OOT_background  = QTable.read(f'Bkg-Spec_{obsid}_OT.fits', format='fits', hdu='SPECTRUM')
		Norm_background = QTable.read(f'Bkg-Spec_{obsid}.fits',    format='fits', hdu='SPECTRUM')
		Norm_background.replace_column(name='COUNTS', col=(Norm_background['COUNTS']-OOT_background['COUNTS']*OOT_fraction))
		spec_hdr        = fits.getheader(filename=f'Bkg-Spec_{obsid}.fits', extname='SPECTRUM').copy()
		fits.update(filename=f'Bkg-Spec_{obsid}.fits', extname='SPECTRUM', data=Norm_background.as_array(), header=spec_hdr)
		# Pile-up correction & channel grouping
		MyTask('rmfgen', [f"spectrumset=Src-Spec_{obsid}.fits", f"rmfset=RMF_{obsid}.rmf", 'raweventfile=events04.dat',
						  'format=var', 'detmaptype=psf', 'correctforpileup=true']).run()
		MyTask('arfgen', [f"spectrumset=Src-Spec_{obsid}.fits", f"arfset=ARF_{obsid}.arf", 'withrmfset=true', f"rmfset=RMF_{obsid}.rmf",
						  'detmaptype=psf', 'filterdss=true', 'useodfatt=true', 'withbadpixcorr=true', 'setbackscale=true',
						  f"badpixlocation=Normal-EL_{obsid}.fits"]).run()
		heasptools.ftmarfrmf(rmffile=f"RMF_{obsid}.rmf", arffile=f"ARF_{obsid}.arf", outfile=f"RSP_{obsid}.rsp",
							 telescop="XMM-Newton", instrume="EPIC-pn", clobber=True)
		MyTask('specgroup', [f'spectrumset=Src-Spec_{obsid}.fits', f'backgndset=Bkg-Spec_{obsid}.fits', f'rmfset=RMF_{obsid}.rmf', f'arfset=ARF_{obsid}.arf',
							 'overwrite=yes', 'units=KEV', 'oversample=3', 'mincounts=25', 'addfilenames=true', 'setbad=CCF']).run()
		# Moving files
		shutil.copyfile(os.path.join(Workdir,f'Src-Spec_{obsid}.fits'), os.path.join(spectra_dir,f'Src-Spec_{obsid}.fits'))
		shutil.copyfile(os.path.join(Workdir,f'Bkg-Spec_{obsid}.fits'), os.path.join(spectra_dir,f'Bkg-Spec_{obsid}.fits'))
		shutil.copyfile(os.path.join(Workdir,f'ARF_{obsid}.arf'),       os.path.join(spectra_dir,f'ARF_{obsid}.arf'))
		shutil.copyfile(os.path.join(Workdir,f'RMF_{obsid}.rmf'),       os.path.join(spectra_dir,f'RMF_{obsid}.rmf'))
		shutil.copyfile(os.path.join(Workdir,f'RSP_{obsid}.rsp'),       os.path.join(spectra_dir,f'RSP_{obsid}.rsp'))
		shutil.copyfile(os.path.join(Workdir,f'SED_{obsid}.fits'),      os.path.join(spectra_dir,f'SED_{obsid}.fits'))
		shutil.copyfile(os.path.join(Workdir,f'Spectrum_{obsid}.gif'),  os.path.join(spectra_dir,f'Spectrum_{obsid}.gif'))
		#shutil.copyfile(os.path.join(Workdir,f'Image-PN_{obsid}_overlayed.png'), os.path.join(image_dir,f'Image-PN_{obsid}_overlayed.png'))
		
		# Step 10: Source event list 
		#MyTask('evselect', [f"table=Normal-EL_{obsid}.fits", f"expression=#XMMEA_EP && (PATTERN<=4) && ({Source_Region})",
		#			   "keepfilteroutput=yes", "writedss=true", "updateexposure=true", "filterexposure=true"]).run()
		#shutil.copyfile(os.path.join(Workdir,f'Normal-EL_{obsid}.fits'), os.path.join(persistent_folder,f'EVL-PN_{obsid}.fits'))
	
	print("-------------------------------- FINISHED! --------------------------------")

In [ ]:
#XMM_Newton(object_name="NGC4051", ObsIDs=[('0903580301',106.83)])
#XMM_Newton(object_name="NGC3516", ObsIDs=[('0912430101',62.12), ('0912430201',58.98), ('0923310601',65.25), ('0854591101',39.03)])
XMM_Newton(object_name="NGC3516", ObsIDs=[('0923310601',65.25), ('0854591101',39.03)])
#XMM_Newton(object_name="Mrk509",  ObsIDs=[''])